# Step 2: Build the feature table (24 conversations)
Line up voice + face features with the engagement score, cut each recording into **1-second windows** (25 frames) and average each window into one row. The same windows also get the w2v-BERT 2.0 and VideoMAE embeddings (averaged over the same seconds) and the 25 raw frames (for the sequence models).
Built by `tools/data24/build_features24.py` and `tools/data24/build_sequences.py`.
*This notebook checks the saved tables and re-computes one window from the raw streams. It does not rebuild them (that takes a long time); to rebuild run `python tools/data24/build_features24.py` and `python tools/data24/build_sequences.py`.*

**In short** (what this notebook shows)

- **What:** Builds the feature table: one row per second (mean of the 25 frames of 88 voice + 714 face values), with engagement, language, session and role; also the 25-frame sequences.
- **Needs the NoXi data?** Yes. The 24-conversation tables were built with tools/data24/build_features24.py and build_sequences.py (the sequences alone are about 4 GB).
- **Result / use:** Result: 48,486 one-second windows, 797 features (88 voice + 709 face). Use it to explain the windows.

### Dataset
**NoXi** (NOvice eXpert Interaction) - expert-novice conversations with continuous engagement labels.
- Paper: A. Cafaro et al., ACM ICMI 2017 (https://doi.org/10.1145/3136755.3136780); challenge page: https://multimediate.perceptualui.org
- Used here: **24 conversations** (8 English, 8 French, 8 German; 48 participant recordings; 48,486 one-second windows).
- Licence: NoXi is available for research under an End-User Licence Agreement. The data is **not included** and must not be shared.

Go to the project folder.

In [2]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

**Imports.**

In [3]:
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
pd.set_option('display.width', 200)

**The feature table.** One row per one-second window: 88 eGeMAPS values, 714 OpenFace values, the engagement label, language, session, role and the second.

In [4]:
df = pd.read_parquet('data/features/noxi_engagement.parquet')
voice = [c for c in df.columns if c.startswith('voice_')]
face = [c for c in df.columns if c.startswith('face_')]
print('rows (windows):', len(df), '| columns:', df.shape[1])
print('voice columns:', len(voice), '| face columns:', len(face))
print('other columns:', [c for c in df.columns if c not in voice + face])

rows (windows): 48486 | columns: 807
voice columns: 88 | face columns: 714
other columns: ['engagement', 'language', 'session', 'role', 'second']


**Windows per language.** 8 conversations and 16 recordings per language.

In [5]:
g = df.groupby('language').agg(conversations=('session', 'nunique'), recordings=('session', lambda s: 0), windows=('session', 'size'),
                             mean_engagement=('engagement', 'mean'), std=('engagement', 'std'))
g['recordings'] = df.groupby('language').apply(lambda x: x[['session', 'role']].drop_duplicates().shape[0])
g.loc['Total'] = [g.conversations.sum(), g.recordings.sum(), g.windows.sum(), df.engagement.mean(), df.engagement.std()]
print(g.round(3).to_string())

          conversations  recordings  windows  mean_engagement    std
language                                                            
English             8.0        16.0  15074.0            0.541  0.141
French              8.0        16.0  19060.0            0.601  0.156
German              8.0        16.0  14352.0            0.578  0.139
Total              24.0        48.0  48486.0            0.576  0.148


**Features used by the models: 797.** The first 5 OpenFace columns (frame number, face id, time stamp, confidence, success) describe the recording, not behaviour, so they are dropped: 88 voice + 709 face.

In [6]:
used = voice + [f'face_{i}' for i in range(5, 714)]
print('features used:', len(used), '(88 voice +', len(used) - 88, 'face)')
print('missing values in the used features:', int(df[used].isna().sum().sum()))
print('engagement range:', round(df.engagement.min(), 4), 'to', round(df.engagement.max(), 4))

features used: 797 (88 voice + 709 face)


missing values in the used features: 0
engagement range: 0.0 to 0.9983


**Embeddings and sequences** are stored next to the table, in the same row order. They are opened without loading them into memory.

In [7]:
w2v = np.load('data/features/pretrained_w2vbert2.npy', mmap_mode='r')
vmae = np.load('data/features/pretrained_videomae.npy', mmap_mode='r')
seq = np.load('data/features/noxi_sequences.npy', mmap_mode='r')
print('w2v-BERT 2.0 :', w2v.shape)
print('VideoMAE     :', vmae.shape)
print('sequences     :', seq.shape, '(windows x 25 frames x 802 values)')
print('same number of rows as the table:', len(df) == len(w2v) == len(vmae) == len(seq))

w2v-BERT 2.0 : (48486, 1024)
VideoMAE     : (48486, 1408)
sequences     : (48486, 25, 802) (windows x 25 frames x 802 values)
same number of rows as the table: True


**Check one window against the raw streams.** We read the raw 25-frames-per-second streams of one recording, cut out one second, average it, and compare with the row in the table. The difference should be 0.

In [8]:
def read_stream(path):
    # the XML header says how many numbers per frame and their type; the .stream~ file holds the numbers
    info = ET.parse(path).getroot().find('info')
    dim = int(info.get('dim'))
    dtype = np.float32 if info.get('type') == 'FLOAT' else np.float64
    data = np.fromfile(path + '~', dtype=dtype)
    frames = data.size // dim
    return data[: frames * dim].reshape(frames, dim)

session, role = '026', 'expert'
row = df[(df.session == session) & (df.role == role)].iloc[20]
start = int(row.second) * 25
folder = os.path.join('data', 'noxi', session)
v = read_stream(os.path.join(folder, f'{role}.audio.egemapsv2.stream'))
f = read_stream(os.path.join(folder, f'{role}.openface2.stream'))
label = pd.to_numeric(pd.read_csv(os.path.join(folder, f'{role}.engagement.annotation.csv'), header=None)[0], errors='coerce').to_numpy()
print('recording', session, role, '| second', int(row.second), '| frames', start, 'to', start + 24)
print('raw streams: voice', v.shape, 'face', f.shape, '| labels', label.shape)
print('voice  max difference:', float(np.abs(np.nanmean(v[start:start + 25], 0) - row[voice].to_numpy(float)).max()))
print('face   max difference:', float(np.abs(np.nanmean(f[start:start + 25], 0) - row[face].to_numpy(float)).max()))
print('label  difference   :', float(abs(np.nanmean(label[start:start + 25]) - row.engagement)))

recording 026 expert | second 20 | frames 500 to 524
raw streams: voice (18082, 88) face (18080, 714) | labels (18083,)
voice  max difference: 0.0


face   max difference: 0.0
label  difference   : 5.207214381908187e-09


**Result.** The table has 48,486 one-second windows with 797 features, the embeddings and sequences line up with it, and a window recomputed from the raw streams matches the table. Step 3 explores the data.